### Customer Churn Classification

In [22]:
#importing libraries

import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder



In [23]:
#loading dataset
filepath = "churn_modelling.csv"
data = pd.read_csv(filepath)
data.head(2)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0


In [24]:
#dropping columns RowNumber, CustomerId, Surname

data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)
data.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [25]:
#checking for null values

data.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [26]:
#checking for duplicate rows

data.duplicated().sum()

np.int64(0)

In [27]:
#checking datatype of features

data.dtypes

CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [28]:
#splitting data into dependent and independent features

y = data['Exited']
x = data.drop(columns=['Exited'])

In [29]:
#splitting data into train and test sets

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 10)
(2000, 10)
(8000,)
(2000,)


In [30]:
#encoding categorical features

x_train['Gender'].value_counts()

Gender
Male      4362
Female    3638
Name: count, dtype: int64

In [31]:
#using labelencoder to encode feature Gender--Male 1 and Female 0

label_encoder = LabelEncoder()

x_train['Gender'] = label_encoder.fit_transform(x_train['Gender'])
x_train['Gender'].value_counts()

Gender
1    4362
0    3638
Name: count, dtype: int64

In [32]:
x_test['Gender'] = label_encoder.transform(x_test['Gender'])
x_test['Gender'].value_counts()

Gender
1    1095
0     905
Name: count, dtype: int64

In [33]:
#encoding feature Geography

data['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [34]:
#applying one hot encoding on feature Geography

onehot_encoder = OneHotEncoder()

geography_train_arr = onehot_encoder.fit_transform(x_train[['Geography']]).toarray()
geography_train_arr

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.]], shape=(8000, 3))

In [35]:
geography_test_arr = onehot_encoder.transform(x_test[['Geography']]).toarray()
geography_test_arr

array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       ...,
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 1., 0.]], shape=(2000, 3))

In [36]:
geography_train_df = pd.DataFrame(geography_train_arr, columns=onehot_encoder.get_feature_names_out())
geography_train_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
...,...,...,...
7995,1.0,0.0,0.0
7996,1.0,0.0,0.0
7997,1.0,0.0,0.0
7998,1.0,0.0,0.0


In [37]:
geography_test_df = pd.DataFrame(geography_test_arr, columns=onehot_encoder.get_feature_names_out())
geography_test_df

,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,0.0,1.0,0.0
4,0.0,0.0,1.0
...,...,...,...
1995,0.0,1.0,0.0
1996,1.0,0.0,0.0
1997,1.0,0.0,0.0
1998,1.0,0.0,0.0


In [38]:
x_train = pd.concat([x_train.reset_index(drop=True), geography_train_df.reset_index(drop=True)], axis=1)
x_train

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,686,France,1,32,6,0.00,2,1,1,179093.26,1.0,0.0,0.0
1,632,Germany,1,42,4,119624.60,2,1,1,195978.86,0.0,1.0,0.0
2,559,Spain,1,24,3,114739.92,1,1,0,85891.02,0.0,0.0,1.0
3,561,France,0,27,9,135637.00,1,1,0,153080.40,1.0,0.0,0.0
4,517,France,1,56,9,142147.32,1,0,0,39488.04,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,768,France,1,54,8,69712.74,1,1,1,69381.05,1.0,0.0,0.0
7996,682,France,0,58,1,0.00,1,1,1,706.50,1.0,0.0,0.0
7997,735,France,0,38,1,0.00,3,0,0,92220.12,1.0,0.0,0.0
7998,667,France,1,43,8,190227.46,1,1,0,97508.04,1.0,0.0,0.0


In [39]:
x_test = pd.concat([x_test.reset_index(drop=True), geography_test_df.reset_index(drop=True)], axis=1)
x_test

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,596,Germany,1,32,3,96709.07,2,0,0,41788.37,0.0,1.0,0.0
1,623,France,1,43,1,0.00,2,1,1,146379.30,1.0,0.0,0.0
2,601,Spain,0,44,4,0.00,2,1,0,58561.31,0.0,0.0,1.0
3,506,Germany,1,59,8,119152.10,2,1,1,170679.74,0.0,1.0,0.0
4,560,Spain,0,27,7,124995.98,1,1,1,114669.79,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,602,Germany,0,53,5,98268.84,1,0,1,45038.29,0.0,1.0,0.0
1996,609,France,1,25,10,0.00,1,0,1,109895.16,1.0,0.0,0.0
1997,730,France,0,47,7,0.00,1,1,0,33373.26,1.0,0.0,0.0
1998,692,France,1,29,4,0.00,1,1,0,76755.99,1.0,0.0,0.0


In [40]:
#dropping feature Geography

x_train = x_train.drop(['Geography'], axis=1)
x_test = x_test.drop(['Geography'], axis=1)

In [41]:
#standardizing data

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [43]:
#saving objects scaler, onehot_encoder and label_encoder in pkl files.

with open("label_encoder.pkl", "wb") as file_obj:
    pickle.dump(label_encoder, file_obj)

with open("onehot_encoder", "wb") as file_obj:
    pickle.dump(onehot_encoder, file_obj)

with open("standard_scaler.pkl", "wb") as file_obj:
    pickle.dump(scaler, file_obj)        